# 00 — Setup and Data Audit

Environment setup, dependency check, configuration loading, global mode
selection, and integrity audits. Every other notebook in this repository
assumes this notebook has been run first in the same session (or that its
outputs -- `results/environment_manifest.json` -- already exist).

**Modes** (see the next cell): `RUN_MODE` (`quick`/`full`) controls
computation scale; `DATA_MODE` (`public`/`private`) controls which data
inputs are used. `DATA_MODE="public"` never requires customer names, raw
customer IDs, exact private GPS coordinates, or confidential operational
records.


In [ ]:
# --- Global controls ---
# When this notebook is run STANDALONE, these defaults apply. When it is
# run as part of 99_REPRODUCE_ALL.ipynb's sequencer, RUN_MODE/DATA_MODE
# are already set by that notebook's own top cell BEFORE this cell runs --
# in that case, this cell must NOT overwrite them (that was a real bug:
# the requested mode was silently reset back to the default here).
if 'RUN_MODE' not in dir():
    RUN_MODE = "quick"    # "quick" (smoke test) or "full" (research computation)
if 'DATA_MODE' not in dir():
    DATA_MODE = "public"  # "public" (de-identified) or "private" (authorized operational inputs)

assert RUN_MODE in ("quick", "full")
assert DATA_MODE in ("public", "private")
print(f"RUN_MODE={RUN_MODE}, DATA_MODE={DATA_MODE}")


## Environment detection and PROJECT_ROOT

Detects Colab vs local execution. On Colab, mounts Google Drive and uses a
persistent project folder there. Locally, uses a folder relative to this
notebook. **Edit `LOCAL_PROJECT_ROOT` below if running outside Colab.**


In [ ]:
import os, sys, subprocess

# --- Configure these two if running on a fresh Colab runtime ---
# REPO_URL: fill this in with the GitHub URL once the repository has been
# published (e.g. "https://github.com/<org>/IJIES_Reproducible_Routing.git").
# Left empty here because no public URL exists yet at packaging time.
REPO_URL = ""
REPO_DIRNAME = "IJIES_Reproducible_Routing"

def in_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

IN_COLAB = in_colab()
print(f"Running in Colab: {IN_COLAB}")

def looks_like_repo_root(path):
    """A directory is considered a valid checkout if it has the src/data.py
    marker file -- cheap, specific, and avoids false positives on an empty
    or unrelated folder."""
    return os.path.exists(os.path.join(path, "src", "data.py"))

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PARENT = "/content/drive/MyDrive/IJIES_Routing_Reproducibility"
    os.makedirs(DRIVE_PARENT, exist_ok=True)

    # 1) Already checked out directly at DRIVE_PARENT?
    if looks_like_repo_root(DRIVE_PARENT):
        REPO_ROOT = DRIVE_PARENT
    # 2) Checked out as a subfolder of DRIVE_PARENT (e.g. after a manual
    #    upload or a previous clone)?
    elif looks_like_repo_root(os.path.join(DRIVE_PARENT, REPO_DIRNAME)):
        REPO_ROOT = os.path.join(DRIVE_PARENT, REPO_DIRNAME)
    # 3) Not found -- clone it, if a URL has been configured.
    elif REPO_URL:
        target = os.path.join(DRIVE_PARENT, REPO_DIRNAME)
        print(f"No existing checkout found; cloning {REPO_URL} into {target} ...")
        subprocess.run(["git", "clone", REPO_URL, target], check=True)
        REPO_ROOT = target
    else:
        raise FileNotFoundError(
            "No repository checkout found under Google Drive, and REPO_URL is "
            "not set. Either (a) set REPO_URL above to this repository's GitHub "
            "URL once it has been published and re-run this cell, or (b) "
            "manually upload/clone the repository into "
            f"'{DRIVE_PARENT}' (or a subfolder of it) before running this "
            "notebook.")
    PROJECT_ROOT = DRIVE_PARENT
else:
    # Local execution: assume this notebook lives at <repo>/notebooks/.
    candidate = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if looks_like_repo_root(candidate):
        REPO_ROOT = candidate
    else:
        raise FileNotFoundError(
            f"'{candidate}' does not look like a repository checkout (no "
            "src/data.py found). Run this notebook from within the "
            "repository's notebooks/ folder, or edit REPO_ROOT manually.")
    PROJECT_ROOT = REPO_ROOT

sys.path.insert(0, REPO_ROOT)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"REPO_ROOT = {REPO_ROOT}")


## Dependencies: install/check, and record versions

In [ ]:
import subprocess, importlib

REQUIRED_PACKAGES = ["pandas", "numpy", "scipy", "matplotlib"]
for pkg in REQUIRED_PACKAGES:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import pandas, numpy, scipy, matplotlib
import platform, json as _json

versions = {
    "python": platform.python_version(),
    "pandas": pandas.__version__,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
}
print(_json.dumps(versions, indent=2))

In [ ]:
import random
import numpy as np

GLOBAL_SEED = 101  # matches the main experiment's first seed; individual
                    # stages set their own explicit seeds where documented.
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
print(f"Global seed set: {GLOBAL_SEED}")


## Load configs

In [ ]:
import json, os

CONFIG_DIR = os.path.join(REPO_ROOT, "configs")
configs = {}
for name in ["experiment_config", "sa_config", "fuzzy_config", "liu_alns_config"]:
    path = os.path.join(CONFIG_DIR, f"{name}.json")
    with open(path) as f:
        configs[name] = json.load(f)
    print(f"Loaded {name}.json")

experiment_config = configs["experiment_config"]
sa_config = configs["sa_config"]
fuzzy_config = configs["fuzzy_config"]
liu_alns_config = configs["liu_alns_config"]


## Run mandatory split-stop regression tests

These MUST pass before any downstream notebook is trusted. See
`tests/test_split_stop_regression.py` and `src/data.py` for the full
bugfix rationale.


In [ ]:
result = subprocess.run([sys.executable, os.path.join(REPO_ROOT, "tests", "test_split_stop_regression.py")],
                        capture_output=True, text=True, cwd=REPO_ROOT)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise AssertionError("Split-stop regression tests FAILED -- do not proceed to downstream notebooks.")
print("Regression tests: PASS")


## Data availability audit (mode-dependent)

In [ ]:
DATA_DEID_DIR = os.path.join(REPO_ROOT, "data_deidentified")
DATA_DEMO_DIR = os.path.join(REPO_ROOT, "data_demo_synthetic")
DATA_PRIVATE_DIR = os.path.join(REPO_ROOT, "data_private")

audit = {"data_mode": DATA_MODE, "checks": {}}

if DATA_MODE == "public":
    # Statistical reproduction inputs: aggregated, de-identified experiment
    # outcomes only (no customer-level operational data, no locked routes,
    # no full derived matrices -- see REPRODUCIBILITY_NOTE.md's data-
    # minimization rationale).
    required_experiment_outputs = [
        os.path.join(DATA_DEID_DIR, "experiment_outputs", "nr_fr_rg_sa_block_level.csv"),
        os.path.join(DATA_DEID_DIR, "experiment_outputs", "liu_alns_block_level.csv"),
        os.path.join(DATA_DEID_DIR, "experiment_outputs", "fuzzy_holdout_block_level.csv"),
        os.path.join(DATA_DEID_DIR, "calibration_split.json"),
    ]
    # NOTE: TomTom-derived data (raw or aggregated) is NOT required here --
    # redistribution permission has not been confirmed, so no such file ships
    # in data_deidentified/. Notebooks 02 and 09 handle its absence explicitly.
    # Computational-code-path smoke-testing inputs: fully SYNTHETIC (fake),
    # not derived from any real data -- exercises src/data.py, src/simulator.py,
    # src/search.py, src/liu_alns.py without needing real/de-identified
    # commercial routing data.
    required_synthetic = [
        os.path.join(DATA_DEMO_DIR, "customer_stops", "synthetic_customer_day_stops.csv"),
        os.path.join(DATA_DEMO_DIR, "matrices", "synthetic_distance_matrix.csv"),
        os.path.join(DATA_DEMO_DIR, "matrices", "synthetic_travel_time_p50_matrix.csv"),
        os.path.join(DATA_DEMO_DIR, "synthetic_locked_routes.csv"),
    ]
    for p in required_experiment_outputs + required_synthetic:
        ok = os.path.exists(p)
        audit["checks"][os.path.relpath(p, REPO_ROOT)] = ok
        print(f"{'OK  ' if ok else 'MISS'}  {p}")
    if not all(audit["checks"].values()):
        raise FileNotFoundError("One or more required public files are missing.")
else:
    print(f"DATA_MODE=private: expecting authorized operational inputs under {DATA_PRIVATE_DIR}")
    print("(Not distributed publicly. See README.md Level 2 reproduction notes.)")
    audit["checks"]["private_data_dir_exists"] = os.path.isdir(DATA_PRIVATE_DIR)

print("\nData availability audit:", "PASS" if all(audit['checks'].values()) else "FAIL")


## Calibration/held-out date-split audit (12 + 51 = 63)

In [ ]:
import pandas as pd

# calibration_split.json carries every instance ID actually used in the
# study (63 total: 12 calibration + 51 held-out) -- this is a study-design
# artifact (which dates were used for what), not customer-level data, so
# it is safe to publish and use directly here rather than reading it back
# out of a removed customer-level file.
with open(os.path.join(DATA_DEID_DIR, "calibration_split.json")) as f:
    split = json.load(f)
all_instances = sorted(split['calibration_instances']) + sorted(split['heldout_instances'])
print(f"Total distinct instances found: {len(all_instances)} (expected 63)")
assert len(all_instances) == 63, f"Expected 63 instances, found {len(all_instances)}"

n_calib = experiment_config["calibration_dates_count"]
n_holdout = experiment_config["holdout_dates_count"]
print(f"Documented split: {n_calib} calibration + {n_holdout} held-out = {n_calib + n_holdout}")
assert n_calib + n_holdout == 63
assert len(split['calibration_instances']) == n_calib
assert len(split['heldout_instances']) == n_holdout

print("Date-split audit: PASS")


## Environment manifest (written for downstream notebooks / audit trail)

In [ ]:
import datetime

manifest = {
    "run_mode": RUN_MODE,
    "data_mode": DATA_MODE,
    "project_root": PROJECT_ROOT,
    "repo_root": REPO_ROOT,
    "in_colab": IN_COLAB,
    "package_versions": versions,
    "global_seed": GLOBAL_SEED,
    "regression_tests": "PASS",
    "data_availability": audit,
    "date_split_audit": "PASS",
    "timestamp": datetime.datetime.now().isoformat(),
}

results_dir = os.path.join(REPO_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)
manifest_path = os.path.join(results_dir, "environment_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Saved {manifest_path}")


## Status

In [ ]:
print("=" * 60)
NOTEBOOK_00_STATUS = "PASS"
print(f"NOTEBOOK 00 STATUS: {NOTEBOOK_00_STATUS}")
print("=" * 60)
print(json.dumps(manifest, indent=2))
